In [16]:
from selenium import webdriver
from selenium.webdriver.common.by import By
from selenium.webdriver.chrome.service import Service
from selenium.webdriver.support.ui import WebDriverWait
from selenium.webdriver.support import expected_conditions as EC
from selenium.common.exceptions import TimeoutException
from webdriver_manager.chrome import ChromeDriverManager
from bs4 import BeautifulSoup
import time

driver = webdriver.Chrome(service=Service(ChromeDriverManager().install()))
html_source = ""

try:
    driver.maximize_window()
    url = 'https://search.naver.com/search.naver?where=image&mode=column&section=image&query=%EB%A7%A4%EC%9A%B4%EC%A0%9C%EC%9C%A1%EB%B3%B6%EC%9D%8C&res_fr=0&res_to=0&sm=tab_opt&color=&ccl=0&nso=so:r,p:from20240601to20240630&recent=0&gif=0&optStr=&nso_open=1&pq='
    driver.get(url)

    # '더보기' 버튼을 끝까지 클릭해서 모든 <img> 태그를 생성
    while True:
        try:
            image_count_before = len(driver.find_elements(By.CSS_SELECTOR, "img._fe_image_tab_content_thumbnail_image"))
            driver.execute_script("window.scrollTo(0, document.body.scrollHeight);")
            more_button = WebDriverWait(driver, 3).until(
                EC.element_to_be_clickable((By.CSS_SELECTOR, "a.btn_more"))
            )
            driver.execute_script("arguments[0].click();", more_button)
            time.sleep(1.5)
            image_count_after = len(driver.find_elements(By.CSS_SELECTOR, "img._fe_image_tab_content_thumbnail_image"))
            if image_count_after == image_count_before:
                break
        except TimeoutException:
            break
            
    print("'더보기' 클릭을 완료했습니다.")

    # --- [핵심] 모든 이미지를 강제로 로딩시키기 위한 최종 스크롤 ---
    print("생성된 모든 이미지의 로딩을 위해 최종 스크롤을 시작합니다...")
    # 페이지 전체 높이를 가져옴
    total_height = driver.execute_script("return document.body.scrollHeight")
    # 화면 높이(800px)만큼 여러 번 스크롤
    for i in range(int(total_height / 800) + 1):
        driver.execute_script(f"window.scrollTo(0, {i * 800});")
        time.sleep(0.3) # 각 스크롤 후 짧은 대기
    print("최종 스크롤 완료.")
    # -----------------------------------------------------------

    html_source = driver.page_source

finally:
    print("브라우저를 닫습니다.")
    driver.quit()

if html_source:
    soup = BeautifulSoup(html_source, 'html.parser')
    images = soup.select('img._fe_image_tab_content_thumbnail_image')
    
    print(f"\n총 {len(images)}개의 <img> 태그를 찾았습니다.")

    valid_image_urls = set()
    for img in images:
        img_url = img.get('src')
        if img_url and img_url.startswith('https://'):
            valid_image_urls.add(img_url)

    print(f"고유하고 유효한 이미지 URL 개수: {len(valid_image_urls)}")

'더보기' 클릭을 완료했습니다.
생성된 모든 이미지의 로딩을 위해 최종 스크롤을 시작합니다...
최종 스크롤 완료.
브라우저를 닫습니다.

총 250개의 <img> 태그를 찾았습니다.
고유하고 유효한 이미지 URL 개수: 169


In [21]:
import joblib
import pandas as pd

# 1) label_map 로드
path = r"C:\ksmindpks\deep-dish\models\label_to_index_20250716_144252.joblib"
label_map = joblib.load(path)

# 2) 키 리스트 생성
keys_list = list(label_map.keys())

# 3) DataFrame으로 변환
df = pd.DataFrame({'label': keys_list})

# 4) 엑셀로 저장
out_excel = r"C:\ksmindpks\deep-dish\models\labels.xlsx"
df.to_excel(out_excel, index=False, encoding='utf-8-sig', sheet_name='Labels')

print(f"✔️ 저장 완료: {out_excel} ({len(keys_list)}개 레이블)")


C:\Users\user\anaconda3\lib\site-packages\pandas\util\_decorators.py:211: FutureWarning: the 'encoding' keyword is deprecated and will be removed in a future version. Please take steps to stop the use of 'encoding'
  return func(*args, **kwargs)


✔️ 저장 완료: C:\ksmindpks\deep-dish\models\labels.xlsx (84개 레이블)
